# CHAI — Appariement AMR *fairness* → document (all manifestos)

Ce notebook produit, pour un tirage aléatoire de phrases du corpus **fairness-MapAIE**,
un appariement **fiable et vérifié** de chaque graphe AMR de *fairness* au document
dont il est issu, et au type d'acteurice correspondant dans la feuille *all manifestos*
(`AI_Ethics_Manifestos.xlsx`).

**Chaîne d'appariement**

```
phrase  --(sample_id)-->  AMR parsé  --(::id + ::doc_id)-->  doc_id
doc_id  = numéro de ligne (1-indexé) dans list_urls.txt
        --(URL, clé exacte)-->  ligne du xlsx  -->  Institution / Sector / actor_type
```

**Deux garde-fous, motivés par des erreurs constatées :**

1. *Décalage AMR ↔ phrase (off-by-one).* Un ancien `ego_summary` présentait un `doc_id`
   décalé d'un cran. On l'élimine en **jointure sur `sample_id`** (jamais sur la position
   dans le fichier) + une assertion `doc_id(AMR) == doc_id(échantillon)`.

2. *Appariement doc → manifeste par titre approximatif.* Le titre lu en tête de `.txt`
   + *fuzzy match* peut se tromper. On le remplace par une **clé exacte : l'URL**
   (`doc_id` → `list_urls.txt` → colonne `URL` du xlsx). Le titre reste en secours,
   avec un rapport de couverture.


## 0 — Configuration

In [ ]:
import os, re, json, difflib
import numpy as np
import pandas as pd

# --- mot-cle cible (une seule ligne a changer pour retargeter le pipeline) ---
KEY_WORD     = "fairness"
KEY_CONCEPTS = {"fairness", "fair-01"}   # etiquettes de concept AMR qui realisent KEY_WORD

# --- echantillonnage ---------------------------------------------------------
N_SAMPLE      = 100
MAX_PER_DOC   = 10        # cap souple : evite qu'un doc prolixe domine le tirage
RANDOM_SEED   = 42
MIN_TOK, MAX_TOK = 4, 80  # ecarte fragments et echecs de segmentation

# --- chemins (a ajuster a ton clone local) -----------------------------------
MAPAIE_DIR = "./mapaie"                         # clone gitlab.telecom-paris.fr/tiphaine.viard/mapaie
TXT_DIR    = os.path.join(MAPAIE_DIR, "txt")    # 1 fichier <doc_id>.txt par document
URLS_PATH  = os.path.join(MAPAIE_DIR, "list_urls.txt")
XLSX_PATH  = "AI_Ethics_Manifestos.xlsx"
XLSX_SHEET = "List of documents"

CSV_CORPUS = f"{KEY_WORD}_MapAIE.csv"           # corpus phrase/doc_id (cree en 0.1 si absent)
AMR_OUT    = f"{KEY_WORD}_sample_penmans.txt"   # AMR parses (Stage 3)
SAMPLE_OUT = f"{KEY_WORD}_sample_100.csv"
PAIRED_OUT = f"{KEY_WORD}_amr_doc_paired.csv"   # livrable final

RUN_PARSER = False   # passer a True en local, une fois amrlib + un modele installes

rng = np.random.default_rng(RANDOM_SEED)
print("config OK — KEY_WORD =", KEY_WORD)

## 1 — Corpus fairness-MapAIE

Si `fairness_MapAIE.csv` existe déjà (produit par le notebook `IA_717`), on le charge.
Sinon on le reconstruit à partir des `.txt` : découpage en phrases, on garde celles
qui contiennent `\bfairness\b`. Chaque phrase porte son `doc_id` (= nom du fichier `.txt`).

In [ ]:
def build_corpus_from_txt(txt_dir=TXT_DIR):
    records = []
    for fname in sorted(os.listdir(txt_dir)):
        if not fname.endswith(".txt"):
            continue
        doc_id = int(fname[:-4])
        with open(os.path.join(txt_dir, fname), encoding="utf-8", errors="replace") as f:
            text = f.read()
        for sent in re.split(r"(?<=[.!?])\s+", text):
            if re.search(rf"\b{KEY_WORD}\b", sent, re.IGNORECASE):
                records.append({"doc_id": doc_id, "sentence": sent.strip()})
    return pd.DataFrame(records, columns=["doc_id", "sentence"])

if os.path.exists(CSV_CORPUS):
    raw = pd.read_csv(CSV_CORPUS)
    print(f"corpus charge depuis {CSV_CORPUS}")
else:
    raw = build_corpus_from_txt()
    raw.to_csv(CSV_CORPUS, index=False, escapechar="\\")
    print(f"corpus reconstruit -> {CSV_CORPUS}")

print(f"{len(raw)} phrases / {raw['doc_id'].nunique()} documents")

## 2 — Échantillon de N_SAMPLE phrases (couple avec les AMR)

**Règle de cohérence :** l'échantillon et le fichier AMR forment un *couple apparié*
(`sample_id` ↔ `doc_id` ↔ graphe). On ne **retire** un échantillon que lorsqu'on va le
**parser** (`RUN_PARSER=True`) ou s'il n'existe pas encore ; sinon on **recharge**
l'échantillon qui accompagne les penmans existants. C'est ce qui garantit que le
garde-fou de la section 4 passe.

Nettoyage : on écarte phrases sans mot-clé, trop courtes/longues, références
bibliographiques et URL brutes. Tirage reproductible (`RANDOM_SEED`), cap `MAX_PER_DOC`.

In [ ]:
# La regle : l'echantillon et les AMR sont un COUPLE apparie.
# - On (re)tire l'echantillon uniquement quand on va le parser (RUN_PARSER=True),
#   ou s'il n'existe pas encore.
# - Sinon on RECHARGE l'echantillon qui accompagne les penmans existants,
#   pour garantir sample_id <-> doc_id <-> graphe.
reuse_sample = (not RUN_PARSER) and os.path.exists(SAMPLE_OUT) and os.path.exists(AMR_OUT)

if reuse_sample:
    sample = pd.read_csv(SAMPLE_OUT)[["sample_id", "doc_id", "sentence"]]
    print(f"echantillon RECHARGE depuis {SAMPLE_OUT} (couple avec {AMR_OUT}) : {len(sample)} phrases")
else:
    raw["sentence"] = raw["sentence"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    raw["ntok"]     = raw["sentence"].str.findall(r"\b\w+\b").apply(len)
    has_kw = raw["sentence"].str.contains(rf"\b{KEY_WORD}\b", case=False, regex=True)
    url    = raw["sentence"].str.contains(r"https?:/|\.pdf|ssrn|doi", case=False, regex=True)
    cite   = raw["sentence"].str.contains(r"Proceedings of|\bpp\.\s*\d|\bISBN\b|\bvol\.\s*\d", regex=True)
    keep   = has_kw & raw["ntok"].between(MIN_TOK, MAX_TOK) & ~url & ~cite
    clean  = raw[keep].reset_index(drop=True)
    print(f"raw {len(raw)} -> clean {len(clean)} ; {clean['doc_id'].nunique()} documents")

    pool = clean.sample(frac=1, random_state=RANDOM_SEED)   # melange reproductible
    picked, per_doc = [], {}
    for _, row in pool.iterrows():
        d = row["doc_id"]
        if per_doc.get(d, 0) >= MAX_PER_DOC:
            continue
        picked.append(row); per_doc[d] = per_doc.get(d, 0) + 1
        if len(picked) >= N_SAMPLE:
            break
    sample = pd.DataFrame(picked).reset_index(drop=True)
    sample.insert(0, "sample_id", [f"s{i:03d}" for i in range(len(sample))])
    sample = sample[["sample_id", "doc_id", "sentence"]]
    sample.to_csv(SAMPLE_OUT, index=False)
    print(f"echantillon TIRE : {len(sample)} phrases / {sample['doc_id'].nunique()} documents -> {SAMPLE_OUT}")

sample.head()

## 3 — Parse AMR (local)

`amrlib.load_stog_model()` charge le modèle installé par défaut
(`parse_xfm_bart_base` — le même que celui ayant servi aux penmans existants).

**Point clé de l'appariement :** on écrit `::id` **et** `::doc_id` dans l'en-tête de
chaque graphe. `::id` porte le `sample_id`, jamais un simple compteur — c'est lui qui
garantit la traçabilité phrase → graphe indépendamment de l'ordre du fichier.

In [ ]:
if RUN_PARSER:
    import amrlib
    stog   = amrlib.load_stog_model()                 # parse_xfm_bart_base par defaut
    graphs = stog.parse_sents(sample["sentence"].tolist(), add_metadata=False)
    with open(AMR_OUT, "w", encoding="utf-8") as f:
        for sid, did, g in zip(sample["sample_id"], sample["doc_id"], graphs):
            f.write(f"# ::id {sid}\n# ::doc_id {did}\n# ::snt {sample.loc[sample.sample_id==sid,'sentence'].iloc[0]}\n")
            f.write(g.strip() + "\n\n")
    print(f"parse : {len(graphs)} graphes -> {AMR_OUT}")
else:
    assert os.path.exists(AMR_OUT), (
        f"{AMR_OUT} absent : passe RUN_PARSER=True pour parser localement, "
        "ou depose le fichier penman a cote du notebook.")
    print(f"[RUN_PARSER=False] on reutilise {AMR_OUT} deja present")

## 4 — Chargement des AMR + garde-fou anti-décalage

On relit les graphes avec `penman` (les métadonnées `::id` / `::doc_id` sont préservées),
puis on **joint sur `sample_id`** avec l'échantillon et on **vérifie** que le `doc_id`
lu dans l'AMR est identique à celui de l'échantillon. Toute divergence lève une erreur :
c'est le contrôle qui aurait attrapé l'ancien décalage off-by-one.

In [ ]:
import penman

graphs = penman.load(AMR_OUT)
amr_rows = []
for g in graphs:
    sid = g.metadata.get("id")
    did = g.metadata.get("doc_id")
    amr_rows.append({"sample_id": sid,
                     "doc_id_amr": (int(did) if did is not None and str(did).strip().isdigit() else pd.NA),
                     "amr_penman": penman.encode(g)})
amr = pd.DataFrame(amr_rows)
print(f"{len(amr)} graphes lus depuis {AMR_OUT}")

# jointure sur sample_id (jamais sur la position)
chk = amr.merge(sample[["sample_id", "doc_id"]], on="sample_id", how="left", validate="one_to_one")
missing = chk["doc_id"].isna().sum()
assert missing == 0, f"{missing} sample_id de l'AMR absents de l'echantillon"

bad = chk[chk["doc_id_amr"].notna() & (chk["doc_id_amr"] != chk["doc_id"])]
assert len(bad) == 0, ("DECALAGE AMR <-> phrase detecte :\n"
                        + bad[["sample_id","doc_id_amr","doc_id"]].to_string(index=False))
print("garde-fou OK : doc_id(AMR) == doc_id(echantillon) pour les", len(chk), "graphes")

## 5 — Appariement `doc_id → document` par URL (clé exacte)

`doc_id` est le **numéro de ligne (1-indexé)** dans `list_urls.txt` : la ligne `doc_id`
donne l'URL du document, et cette URL est une **clé exacte** dans la colonne `URL` du xlsx.
On récupère ainsi `Institution` / `Sector`, puis `actor_type` par la grille CHAI.

Le titre lu en tête de `.txt` + *fuzzy match* est conservé **en secours uniquement**
(diagnostic), car il peut se tromper — c'est ce qui avait produit une étiquette erronée
(doc 73 « Samsung » au lieu de CEPEJ).

In [ ]:
# grille CHAI : secteur (base, en) -> type d'acteurice (fr)
SECTOR_TO_ACTOR = {
    "academia":                   "academique",
    "civil society":              "societe civile",
    "international organisation":  "organisation internationale",
    "national authority":         "autorite nationale",
    "private sector":             "secteur prive",
}

def norm_url(u):
    """Normalisation d'URL pour l'appariement exact : schéma/www/query/fragment/slash final."""
    u = str(u).strip().lower()
    u = re.sub(r"^https?://", "", u)
    u = re.sub(r"^www\.", "", u)
    u = u.split("?")[0].split("#")[0].rstrip("/")
    return u

def base_sector(s):
    if not isinstance(s, str):
        return None
    return s.split(">")[0].split(",")[0].strip().lower() or None

# --- table des URLs : doc_id -> URL ------------------------------------------
with open(URLS_PATH, encoding="utf-8") as f:
    urls = [l.strip() for l in f if l.strip()]
docid2url = {i + 1: u for i, u in enumerate(urls)}   # 1-indexe

# --- feuille all manifestos --------------------------------------------------
meta = pd.read_excel(XLSX_PATH, sheet_name=XLSX_SHEET)
meta = meta.rename(columns={"Name of the document": "title", "Sector": "sector",
                            "Institution": "institution", "URL": "url"})
meta = meta.dropna(subset=["title"]).copy()
meta["url_norm"]    = meta["url"].map(norm_url)
meta["title_norm"]  = meta["title"].astype(str).str.lower().str.strip()

# URL -> premiere ligne du xlsx qui la porte
url2idx = {}
for idx, row in meta.iterrows():
    k = row["url_norm"]
    if k and k != "nan":
        url2idx.setdefault(k, idx)
print(f"{len(urls)} URLs ; {len(meta)} lignes xlsx ; {len(url2idx)} URLs distinctes indexees")

In [ ]:
# --- secours : titre lu en tete du .txt + fuzzy match (diagnostic seulement) --
titles = meta["title_norm"].tolist()

def title_from_txt(doc_id, n_lines=15):
    path = os.path.join(TXT_DIR, f"{doc_id}.txt")
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in (next(f, "") for _ in range(n_lines)):
            s = line.strip()
            if len(s) >= 12 and re.search(r"[A-Za-z]", s) and not s.isdigit():
                return s
    return None

def fuzzy_actor(doc_id):
    cand = title_from_txt(doc_id)
    if not cand:
        return (None, None, 0.0)
    hit = difflib.get_close_matches(cand.lower(), titles, n=1, cutoff=0.0)
    if not hit:
        return (None, None, 0.0)
    score = difflib.SequenceMatcher(None, cand.lower(), hit[0]).ratio()
    row = meta.loc[meta["title_norm"] == hit[0]].iloc[0]
    return (row["title"], base_sector(row["sector"]), round(score, 3))

# --- resolution par URL pour chaque doc_id de l'echantillon ------------------
def resolve_by_url(doc_id):
    u = docid2url.get(int(doc_id))
    if u is None:
        return dict(url=None, matched=False, title=None, institution=None,
                    sector_raw=None, actor_type=None)
    idx = url2idx.get(norm_url(u))
    if idx is None:
        return dict(url=u, matched=False, title=None, institution=None,
                    sector_raw=None, actor_type=None)
    row = meta.loc[idx]
    base = base_sector(row["sector"])
    return dict(url=u, matched=True, title=row["title"], institution=row["institution"],
                sector_raw=row["sector"], actor_type=SECTOR_TO_ACTOR.get(base))

recs = []
for did in sorted(sample["doc_id"].unique()):
    r = resolve_by_url(did)
    ft, fs, fsc = fuzzy_actor(did)     # secours / diagnostic
    r.update(doc_id=int(did), fuzzy_title=ft, fuzzy_sector=fs, fuzzy_score=fsc)
    recs.append(r)

doc_map = pd.DataFrame(recs).set_index("doc_id")
n = len(doc_map)
print(f"{n} documents dans l'echantillon")
print(f"  apparies par URL      : {int(doc_map['matched'].sum())}/{n}")
print(f"  actor_type renseigne  : {int(doc_map['actor_type'].notna().sum())}/{n}")
unresolved = doc_map.index[~doc_map["matched"] | doc_map["actor_type"].isna()].tolist()
if unresolved:
    print("  a curer a la main     :", unresolved)
doc_map[["title","institution","sector_raw","actor_type","matched"]].head(10)

### Interlude manuel — corrections ponctuelles

Pour les rares `doc_id` non appariés par URL (URL absente du xlsx, ou variante d'URL),
on renseigne l'`actor_type` à la main ici. Vérifier avec l'en-tête du `.txt`
(`title_from_txt(doc_id)`) et l'URL (`docid2url[doc_id]`).

In [ ]:
MANUAL_OVERRIDES = {
    # doc_id: "actor_type",
    # ex. 73: "organisation internationale",
}
for d, a in MANUAL_OVERRIDES.items():
    if d in doc_map.index:
        doc_map.loc[d, "actor_type"] = a
        doc_map.loc[d, "matched"]    = True
print("overrides appliques :", list(MANUAL_OVERRIDES) or "(aucun)")
still = doc_map.index[doc_map["actor_type"].isna()].tolist()
print("encore sans actor_type :", still or "(aucun)")

## 6 — Table finale : un AMR de *fairness* par ligne, apparié à son document

On joint l'AMR (via `sample_id`) à l'échantillon puis à `doc_map` (via `doc_id`).
Le livrable `fairness_amr_doc_paired.csv` contient, pour chaque phrase :
`sample_id, doc_id, sentence, institution, title, sector_raw, actor_type, url, amr_penman`.

In [ ]:
paired = (sample
    .merge(amr[["sample_id", "amr_penman"]], on="sample_id", how="left", validate="one_to_one")
    .merge(doc_map[["url","title","institution","sector_raw","actor_type","matched"]],
           left_on="doc_id", right_index=True, how="left"))

# controle final : chaque phrase a un AMR, chaque doc un type d'acteurice
n_no_amr   = paired["amr_penman"].isna().sum()
n_no_actor = paired["actor_type"].isna().sum()
print(f"phrases              : {len(paired)}")
print(f"  sans AMR           : {n_no_amr}")
print(f"  sans actor_type    : {n_no_actor}")
print("\nrepartition par type d'acteurice :")
print(paired["actor_type"].value_counts(dropna=False).to_string())

cols = ["sample_id","doc_id","sentence","institution","title","sector_raw","actor_type","url","amr_penman"]
paired[cols].to_csv(PAIRED_OUT, index=False)
print(f"\nlivrable -> {PAIRED_OUT}")
paired[["sample_id","doc_id","institution","actor_type"]].head(10)

## 7 — Contrôle qualité (à vérifier avant le RDV)

Trois vérifications lisibles d'un coup d'œil :
- couverture de l'appariement URL et de l'`actor_type` ;
- un échantillon phrase ↔ institution pour un contrôle humain ;
- rappel : le garde-fou de la section 4 a déjà certifié l'alignement AMR ↔ phrase.

In [ ]:
print("=== Couverture ===")
print(f"apparies par URL        : {int(paired['matched'].fillna(False).sum())}/{len(paired)} phrases")
print(f"actor_type renseigne    : {int(paired['actor_type'].notna().sum())}/{len(paired)} phrases")

print("\n=== Controle humain : 8 phrases au hasard (phrase -> institution / type) ===")
for _, r in paired.sample(min(8, len(paired)), random_state=0).iterrows():
    print(f"[{r['sample_id']} doc {r['doc_id']}] {str(r['sentence'])[:70]!r}")
    print(f"      -> {r['institution']}  |  {r['actor_type']}")

print("\n=== Divergence URL vs titre-fuzzy (diagnostic) ===")
diag = doc_map[doc_map["matched"] & doc_map["actor_type"].notna() & doc_map["fuzzy_sector"].notna()]
disagree = diag[diag.apply(lambda x: SECTOR_TO_ACTOR.get(x["fuzzy_sector"]) != x["actor_type"], axis=1)]
print(f"{len(disagree)} doc(s) ou le titre-fuzzy aurait donne un autre type "
      f"(l'URL fait foi) : {disagree.index.tolist()}")